In [1]:
from functools import partial
from models import train_model, save_model, load_model, SimpleCNN, ResNet
from pathlib import Path
from utils import save_pred_to_csv
from predict import predict_test_ensemble

In [ ]:
model_scnn_stride = partial(SimpleCNN, filters=(32,64,128), n_convs=2, dropout=0.4, use_stride=True)
model_scnn_dilated = partial(SimpleCNN, filters=(32,64,128), n_convs=2, dropout=0.4, dilation=2)
model_resnet_se = partial(ResNet, filters=(32,64,128), blocks_per_stage=2, use_se=True)
model_resnet_lean = partial(ResNet, filters=(32,64,128), blocks_per_stage=1, use_se=False)
models = {
    "scnn_stride":   partial(SimpleCNN, filters=(32,64,128), n_convs=2, dropout=0.4, use_stride=True),
    "scnn_dilated":  partial(SimpleCNN, filters=(32,64,128), n_convs=2, dropout=0.4, dilation=2),
    "resnet_se":     partial(ResNet, filters=(32,64,128), blocks_per_stage=2, use_se=True),
    "resnet_lean":   partial(ResNet, filters=(32,64,128), blocks_per_stage=1, use_se=False),
    "scnn_baseline": partial(SimpleCNN, filters=(32,64,128), n_convs=2, dropout=0.4),
    "scnn_3conv": partial(SimpleCNN, filters=(32,64,128), n_convs=3, dropout=0.4)
}

In [ ]:
for idx in range(5):
    for (n,m) in models.items():
        model = train_model(m, epochs=20, seed=idx)
        save_model(model, f"{n}_seed_{idx}")

In [3]:
def find_file_dir(root, needle):
    for path in Path(root).rglob("*"):
        if path.is_file() and needle in path.name:
            return path
    return None

path = "/Users/dominik/Documents/GitHub/IIVP2026-Group-5/outputs/nn_models/new_models"
combination1 = ["scnn_dilated", "resnet_se", "scnn_baseline", "scnn_3conv"]
combination2 = ["scnn_stride", "scnn_dilated", "resnet_se", "resnet_lean"]
trained1 = []
trained2 = []
for n in combination1:
    trained1.append(load_model(find_file_dir(path, n), weights_only=False))
df1_tta, df_probs1_tta = predict_test_ensemble(trained1, use_tta=True)
df1_no_tta, df_probs1_no_tta = predict_test_ensemble(trained1, use_tta=False)
for n in combination2:
    trained2.append(load_model(find_file_dir(path, n),weights_only=False))
df2_tta, df_probs2_tta = predict_test_ensemble(trained2, use_tta=True)
df2_no_tta, df_probs_no_tta = predict_test_ensemble(trained2, use_tta=False)
save_pred_to_csv(df1_tta, 'df1tta')
save_pred_to_csv(df1_no_tta, 'df1notta')
save_pred_to_csv(df2_tta, 'df2tta')
save_pred_to_csv(df2_no_tta, 'df2notta')

TTA enabled: True, using 5 transforms
TTA enabled: True, using 5 transforms
TTA enabled: True, using 5 transforms
TTA enabled: True, using 5 transforms
TTA enabled: False, using 1 transforms
TTA enabled: False, using 1 transforms
TTA enabled: False, using 1 transforms
TTA enabled: False, using 1 transforms
TTA enabled: True, using 5 transforms
TTA enabled: True, using 5 transforms
TTA enabled: True, using 5 transforms
TTA enabled: True, using 5 transforms
TTA enabled: False, using 1 transforms
TTA enabled: False, using 1 transforms
TTA enabled: False, using 1 transforms
TTA enabled: False, using 1 transforms


PosixPath('/Users/dominik/Documents/GitHub/IIVP2026-Group-5/outputs/csv_results/df2notta.csv')